# 05 — CPC vs IMERG: 4-Distribution Comparison

Interactive exploration of **which distribution fits best** across climate regimes
and **why CPC and IMERG disagree** on the best-fitting distribution.

**Distributions compared:**  Log-normal (k=2) · Gamma (k=2) · Pearson III (k=3) · SHASH (k=3)

**SHASH formula:**  Jones & Pewsey (2009), geosciences τ convention  
CDF: Φ(sinh(arcsinh(x/σ) / τ − ε))  
τ=1 → normal · τ>1 → heavier tails · τ<1 → lighter tails

**Key findings (2001–2020, wet threshold 0.1 mm/day, 7-day mean):**
- CPC  (~13 600 fitted land cells): **~95 % Pearson III · ~5 % SHASH** · <1 % LN/Gamma
- IMERG (~24 900 fitted cells):     **~84 % Pearson III · ~14 % SHASH** · ~2 % LN/Gamma
- Pearson III dominates everywhere; SHASH captures the heaviest-tailed (convective) cells
- IMERG has ~3× more SHASH-winning cells than CPC — satellite resolves extremes that gauges miss

**Kernel:** Python (atmo)

In [ ]:
import os
# Fix: VS Code's pydevd (sys.monitoring) clashes with Cartopy's Cython code on
# Python 3.12+.  This env var disables the code-object validation that triggers
# the AssertionError.  Must be set before any cartopy import.
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

import sys
sys.path.insert(0, '..')

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.stats import rv_continuous
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── SHASH distribution (Jones & Pewsey 2009, Biometrika 96 761) ───────────
# Geosciences / TensorFlow convention: tailweight τ = 1 / δ_J&P
#   τ = 1 → normal (when ε = 0)
#   τ > 1 → heavier tails than normal (leptokurtic)
#   τ < 1 → lighter tails than normal (platykurtic)
# CDF: F(x) = Φ(sinh(arcsinh(x) / τ − ε))
class _SHASHDist(rv_continuous):
    def _pdf(self, x, eps, tailweight):
        u = np.arcsinh(x)
        v = u / tailweight - eps
        z = np.sinh(v)
        return np.cosh(v) / (tailweight * np.sqrt(1.0 + x**2)) * stats.norm.pdf(z)
    def _cdf(self, x, eps, tailweight):
        return stats.norm.cdf(np.sinh(np.arcsinh(x) / tailweight - eps))
    def _argcheck(self, eps, tailweight):
        return tailweight > 0

_shash = _SHASHDist(name='shash')

# ── Distribution registry ─────────────────────────────────────────────────
DIST_NAMES  = ['Log-normal', 'Gamma', 'Pearson III', 'SHASH']
DIST_COLORS = ['#2166ac', '#d6604d', '#4dac26', '#f1a340']   # blue, red, green, orange
DIST_LS     = ['-', '--', '-.', ':']
DIST_IDX    = {n: i for i, n in enumerate(DIST_NAMES)}

# ── Matplotlib style ──────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
})

print('Ready.')

---
## 1. Load both datasets

In [ ]:
# ── Load processed time-series arrays ─────────────────────────────────────
da_cpc   = xr.open_dataarray('../data/processed/cpc/cpc_7day_wet.nc')
da_imerg = xr.open_dataarray('../data/processed/imerg/imerg_7day_wet.nc')

# ── Load pre-computed distribution stats ──────────────────────────────────
ds_cpc   = xr.open_dataset('../output/stats/cpc_distribution_stats.nc')
ds_imerg = xr.open_dataset('../output/stats/imerg_distribution_stats.nc')

MIN_SAMPLES = 30   # minimum wet-day samples required to trust a fit

print('CPC  processed:', dict(da_cpc.sizes))
print('IMERG processed:', dict(da_imerg.sizes))
print()
print('CPC  stats vars:', [v for v in ds_cpc.data_vars if 'best_fit' in v or 'pearson' in v or 'shash' in v])

# ── Quick 4-way breakdown ─────────────────────────────────────────────────
for label, ds_s in [('CPC', ds_cpc), ('IMERG', ds_imerg)]:
    if 'best_fit_4way' not in ds_s:
        print(f'{label}: best_fit_4way not found — re-run pipeline with updated code')
        continue
    b4 = ds_s['best_fit_4way'].values
    fitted = int(np.sum(b4 >= 0))
    print(f'\n{label} best-fit breakdown ({fitted} fitted grid points):')
    for i, name in enumerate(DIST_NAMES):
        n = int(np.sum(b4 == i))
        print(f'  {name:<14}: {n:5d}  ({100*n/max(fitted,1):.1f}%)')

---
## 2. CONUS maps: where does each distribution win?

Helper to set up a CONUS-only Cartopy axes (ocean + Canada + Mexico masked).

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader

def _conus_ax(ax):
    """Configure a Cartopy PlateCarree axes for CONUS-only display."""
    ax.set_extent([-125, -66, 24, 50], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN, facecolor='lightgrey', zorder=4)
    countries = shpreader.natural_earth(resolution='50m', category='cultural',
                                        name='admin_0_countries')
    for rec in shpreader.Reader(countries).records():
        if rec.attributes.get('NAME_EN', '') in ('Canada', 'Mexico'):
            ax.add_geometries([rec.geometry], ccrs.PlateCarree(),
                              facecolor='lightgrey', edgecolor='none', zorder=4)
    ax.add_feature(cfeature.STATES, linewidth=0.4, edgecolor='black', alpha=0.6, zorder=5)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.6, zorder=5)
    return ax

print('_conus_ax() ready.')

In [ ]:
# ── 4-way best-fit: CPC (left) vs IMERG (right) ───────────────────────────
fig, axes = plt.subplots(
    1, 2, figsize=(20, 6),
    subplot_kw={'projection': ccrs.PlateCarree()},
)

cmap4 = mcolors.ListedColormap(DIST_COLORS)
norm4 = mcolors.BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap4.N)

for ax, ds_s, title in zip(axes, [ds_cpc, ds_imerg], ['CPC (gauge-based)', 'IMERG v07 (satellite)']):
    ax = _conus_ax(ax)
    data = ds_s['best_fit_4way'].values.astype(float)
    data[ds_s['n_wetdays'].values < MIN_SAMPLES] = np.nan
    cmap4.set_bad('lightgrey')
    im = ax.pcolormesh(ds_s.lon.values, ds_s.lat.values, data,
                       transform=ccrs.PlateCarree(), cmap=cmap4, norm=norm4)
    cb = fig.colorbar(im, ax=ax, orientation='horizontal', pad=0.03, shrink=0.75,
                      ticks=[0, 1, 2, 3])
    cb.ax.set_xticklabels(DIST_NAMES, fontsize=9)
    cb.set_label('Best-fit distribution (lowest AIC)', fontsize=9)
    ax.set_title(title, fontsize=12, fontweight='bold')

fig.suptitle(
    'Best-fit distribution per grid point (4-way AIC comparison)\n'
    '7-day mean precipitation, 2001–2020, wet threshold 0.1 mm/day',
    fontsize=13, fontweight='bold', y=1.02,
)
plt.tight_layout()
plt.show()

In [ ]:
# ── AIC confidence (margin of victory): CPC vs IMERG ─────────────────────
fig, axes = plt.subplots(
    1, 2, figsize=(20, 6),
    subplot_kw={'projection': ccrs.PlateCarree()},
)

cmap_conf = plt.get_cmap('YlOrRd')
cmap_conf.set_bad('lightgrey')

for ax, ds_s, title in zip(axes, [ds_cpc, ds_imerg], ['CPC (gauge-based)', 'IMERG v07 (satellite)']):
    ax = _conus_ax(ax)
    data = ds_s['aic_confidence'].values.copy()
    data[ds_s['n_wetdays'].values < MIN_SAMPLES] = np.nan
    im = ax.pcolormesh(ds_s.lon.values, ds_s.lat.values, data,
                       transform=ccrs.PlateCarree(), cmap=cmap_conf, vmin=0, vmax=50)
    cb = fig.colorbar(im, ax=ax, orientation='horizontal', pad=0.03, shrink=0.85)
    cb.set_label('ΔAIC: 2nd-best minus best  (higher = more decisive winner)', fontsize=9)
    cb.ax.tick_params(labelsize=8)
    ax.set_title(title, fontsize=12, fontweight='bold')

fig.suptitle(
    'AIC confidence — how decisively the best distribution wins\n'
    'Low confidence regions = distributions are interchangeable',
    fontsize=13, fontweight='bold', y=1.02,
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Where do CPC and IMERG AGREE on best distribution? ───────────────────
# Align grids: IMERG might have slightly different lat/lon after regridding
b_cpc   = ds_cpc['best_fit_4way'].values.astype(float)
b_imerg = ds_imerg['best_fit_4way'].interp_like(
    ds_cpc, method='nearest', kwargs={'fill_value': np.nan}
).values.astype(float)

# b_imerg is already on the CPC grid — use it for the mask too
mask = (ds_cpc['n_wetdays'].values < MIN_SAMPLES) | (b_imerg < 0)

# Agreement: same best-fit distribution
agree = (b_cpc == b_imerg).astype(float)
agree[mask] = np.nan

# Where they disagree, show which dataset chose a "more complex" distribution
# (higher index = more parameters)
cpc_more_complex = (b_cpc > b_imerg).astype(float)   # CPC chose P3 or SHASH vs gamma/lognorm in IMERG
cpc_more_complex[mask] = np.nan

agree_pct = float(np.nanmean(agree) * 100)
print(f'Agreement rate (CPC vs IMERG same best distribution): {agree_pct:.1f}%')

fig, axes = plt.subplots(
    1, 2, figsize=(20, 6),
    subplot_kw={'projection': ccrs.PlateCarree()},
)

# Panel A: agree (1) / disagree (0)
ax = _conus_ax(axes[0])
cmap_agree = mcolors.ListedColormap(['#e08080', '#6ab187'])   # red=disagree, green=agree
cmap_agree.set_bad('lightgrey')
im = ax.pcolormesh(ds_cpc.lon.values, ds_cpc.lat.values, agree,
                   transform=ccrs.PlateCarree(), cmap=cmap_agree, vmin=-0.5, vmax=1.5)
cb = fig.colorbar(im, ax=ax, orientation='horizontal', pad=0.03, shrink=0.7, ticks=[0, 1])
cb.ax.set_xticklabels(['Disagree', 'Agree'], fontsize=10)
ax.set_title(f'CPC vs IMERG: best-fit agreement  ({agree_pct:.0f}% agree)',
             fontsize=11, fontweight='bold')

# Panel B: AIC confidence difference (CPC - IMERG), both interpolated to same grid
ax = _conus_ax(axes[1])
conf_imerg_aligned = ds_imerg['aic_confidence'].interp_like(
    ds_cpc, method='nearest', kwargs={'fill_value': np.nan}
).values
conf_diff = ds_cpc['aic_confidence'].values - conf_imerg_aligned
conf_diff[mask] = np.nan
cmap_diff = plt.get_cmap('RdBu_r')
cmap_diff.set_bad('lightgrey')
im = ax.pcolormesh(ds_cpc.lon.values, ds_cpc.lat.values, conf_diff,
                   transform=ccrs.PlateCarree(), cmap=cmap_diff, vmin=-30, vmax=30)
cb = fig.colorbar(im, ax=ax, orientation='horizontal', pad=0.03, shrink=0.85)
cb.set_label('AIC confidence: CPC − IMERG  (blue = CPC more decisive)', fontsize=9)
cb.ax.tick_params(labelsize=8)
ax.set_title('Where is one dataset more decisive than the other?',
             fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 3. Fitting functions (all 4 distributions)

Run once. All regional and single-location cells below call `fit_point()`.

In [ ]:
DISTRIBUTIONS = {
    'Log-normal': {
        'dist': stats.lognorm, 'fit_args': (), 'fit_kwargs': {'floc': 0}, 'k': 2,
        'color': DIST_COLORS[0], 'ls': DIST_LS[0],
    },
    'Gamma': {
        'dist': stats.gamma,   'fit_args': (), 'fit_kwargs': {'floc': 0}, 'k': 2,
        'color': DIST_COLORS[1], 'ls': DIST_LS[1],
    },
    'Pearson III': {
        'dist': stats.pearson3, 'fit_args': (), 'fit_kwargs': {}, 'k': 3,
        'color': DIST_COLORS[2], 'ls': DIST_LS[2],
    },
    'SHASH': {
        # Initial guesses: (eps=0.5, tailweight=1.5)  — J&P formula, τ convention
        'dist': _shash, 'fit_args': (0.5, 1.5), 'fit_kwargs': {'floc': 0}, 'k': 3,
        'color': DIST_COLORS[3], 'ls': DIST_LS[3],
    },
}


def fit_point(lat, lon, da, ds_stats=None, wet_threshold=0.1, min_samples=30):
    """
    Extract wet-day time series at nearest grid point and fit all 4 distributions.

    Returns dict:
      wet       : array of wet-day values
      lat, lon  : actual grid coordinates (nearest-neighbour snapped)
      n         : number of wet-day samples
      fits      : {dist_name: {params, aic, ks_stat, ks_pval, log_likelihood}}
      best      : name of best-fit distribution
      precomputed: values from stats NetCDF (or None)
    """
    ts  = da.sel(lat=lat, lon=lon, method='nearest')
    raw = ts.values
    wet = raw[np.isfinite(raw) & (raw >= wet_threshold)]

    result = {
        'wet': wet, 'lat': float(ts.lat), 'lon': float(ts.lon),
        'n': len(wet), 'fits': {}, 'best': None, 'precomputed': None,
    }

    if len(wet) < min_samples:
        return result

    # Pull pre-computed AIC from stats file if available
    if ds_stats is not None and 'best_fit_4way' in ds_stats:
        pt = ds_stats.sel(lat=lat, lon=lon, method='nearest')
        result['precomputed'] = {k: float(pt[k]) for k in pt.data_vars}

    for name, spec in DISTRIBUTIONS.items():
        try:
            params = spec['dist'].fit(wet, *spec['fit_args'], **spec['fit_kwargs'])
            if name == 'SHASH' and (params[1] <= 0 or params[3] <= 0):
                raise ValueError('invalid SHASH params')
            ll  = float(np.sum(spec['dist'].logpdf(wet, *params)))
            aic = 2 * spec['k'] - 2 * ll
            ks, pv = stats.kstest(wet, spec['dist'].cdf, args=params)
            result['fits'][name] = {
                'params': params, 'log_likelihood': ll,
                'aic': aic, 'ks_stat': ks, 'ks_pval': pv,
            }
        except Exception:
            pass

    if result['fits']:
        result['best'] = min(result['fits'], key=lambda n: result['fits'][n]['aic'])

    return result


def _x_range(wet, extend=1.1):
    return np.linspace(max(wet.min() * 0.3, 0.01), wet.max() * extend, 500)


def print_fit_table(r, label=''):
    """Print AIC/KS table for one fit_point() result."""
    header = f"  {label}  |  n={r['n']}  mean={r['wet'].mean():.2f}  "\
             f"std={r['wet'].std():.2f}  skew={stats.skew(r['wet']):.2f}"
    print(header)
    print('  ' + '-' * 60)
    print(f"  {'Distribution':<14} {'AIC':>9} {'KS stat':>8} {'KS p-val':>9}")
    for name, f in sorted(r['fits'].items(), key=lambda x: x[1]['aic']):
        mark = ' ◄' if name == r['best'] else ''
        print(f"  {name:<14} {f['aic']:>9.1f} {f['ks_stat']:>8.4f} {f['ks_pval']:>9.4f}{mark}")
    print()


print('Fitting functions ready.')

---
## 4. Regional climate regime analysis

Six representative locations across CONUS climate zones.  
For each, fit all 4 distributions to both CPC and IMERG.

In [ ]:
# ── Representative locations ───────────────────────────────────────────────
LOCATIONS = [
    {'lat': 47.5,  'lon': -122.3, 'label': 'Seattle',        'region': 'Pacific NW (maritime)'},
    {'lat': 39.0,  'lon': -94.5,  'label': 'Kansas City',    'region': 'Central Plains (convective)'},
    {'lat': 25.8,  'lon': -80.2,  'label': 'Miami',          'region': 'Subtropical (tropical moisture)'},
    {'lat': 33.5,  'lon': -112.0, 'label': 'Phoenix',        'region': 'Arid SW (sparse/intense)'},
    {'lat': 44.0,  'lon': -69.8,  'label': 'Maine',          'region': 'NE Coast (frontal)'},
    {'lat': 36.0,  'lon': -86.5,  'label': 'Nashville',      'region': 'SE Inland (warm-season convection)'},
]

# Fit both datasets at all locations
results = []
for loc in LOCATIONS:
    r_cpc   = fit_point(loc['lat'], loc['lon'], da_cpc,   ds_cpc)
    r_imerg = fit_point(loc['lat'], loc['lon'], da_imerg, ds_imerg)
    results.append({'loc': loc, 'cpc': r_cpc, 'imerg': r_imerg})

# ── Summary table ─────────────────────────────────────────────────────────
print(f"{'Location':<18} {'Region':<32}  "
      f"{'CPC best':>12}  {'CPC AIC':>9}  "
      f"{'IMERG best':>12}  {'IMERG AIC':>10}  {'Agree?':>7}")
print('-' * 110)
for entry in results:
    loc = entry['loc']
    rc  = entry['cpc']
    ri  = entry['imerg']
    bc  = rc.get('best', '—')
    bi  = ri.get('best', '—')
    ac  = rc['fits'][bc]['aic'] if bc in rc.get('fits', {}) else float('nan')
    ai  = ri['fits'][bi]['aic'] if bi in ri.get('fits', {}) else float('nan')
    agree = '✓' if bc == bi else '✗'
    print(f"{loc['label']:<18} {loc['region']:<32}  "
          f"{bc:>12}  {ac:>9.1f}  "
          f"{bi:>12}  {ai:>10.1f}  {agree:>7}")

In [ ]:
# ── PDF comparison: all 4 distributions × 6 locations × CPC + IMERG ──────
# Layout: 6 rows × 2 cols  (left=CPC, right=IMERG)

n_loc = len(results)
fig, axes = plt.subplots(n_loc, 2, figsize=(14, 4.2 * n_loc), squeeze=False)

for row, entry in enumerate(results):
    loc = entry['loc']
    for col, (r, ds_label) in enumerate([(entry['cpc'], 'CPC'), (entry['imerg'], 'IMERG')]):
        ax = axes[row, col]
        wet = r['wet']
        if len(wet) < MIN_SAMPLES:
            ax.text(0.5, 0.5, 'insufficient data', ha='center', va='center',
                    transform=ax.transAxes, fontsize=10, color='grey')
            ax.set_visible(True)
            continue

        x = _x_range(wet)

        ax.hist(wet, bins=35, density=True,
                color='#c8dff0' if ds_label == 'CPC' else '#ffe5b4',
                edgecolor='white', linewidth=0.3, label='Empirical', zorder=1)

        for name, spec in DISTRIBUTIONS.items():
            if name not in r['fits']:
                continue
            f   = r['fits'][name]
            pdf = spec['dist'].pdf(x, *f['params'])
            lw  = 2.5 if name == r['best'] else 1.5
            lbl = f"{name} (AIC={f['aic']:.0f})" + (' ★' if name == r['best'] else '')
            ax.plot(x, pdf, color=spec['color'], ls=spec['ls'], lw=lw,
                    label=lbl, zorder=2)

        skew_val = stats.skew(wet)
        ax.set_title(
            f"{loc['label']} — {ds_label}  |  n={r['n']}  skew={skew_val:.2f}",
            fontsize=9, fontweight='bold',
        )
        ax.set_xlabel('7-day mean precip (mm/day)', fontsize=8)
        ax.set_ylabel('Density', fontsize=8)
        ax.legend(fontsize=7, loc='upper right', framealpha=0.9)
        ax.set_xlim(left=0)
        ax.tick_params(labelsize=7)

# Column headers
axes[0, 0].set_title('CPC (gauge-based)\n' + axes[0, 0].get_title(),
                      fontsize=10, fontweight='bold')
axes[0, 1].set_title('IMERG v07 (satellite)\n' + axes[0, 1].get_title(),
                      fontsize=10, fontweight='bold')

fig.suptitle(
    '4-distribution PDF fit — 6 climate regions  (★ = best AIC)',
    fontsize=13, fontweight='bold', y=1.005,
)
plt.tight_layout()
plt.show()

---
## 5. AIC comparison bar chart — all distributions × all locations

Visualise the AIC *differences* relative to the best distribution.  
Easier to read than raw AIC values: a bar of height 0 is the winner; taller = worse.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=False)
x_ticks = np.arange(len(results))
width    = 0.2

for ax, ds_label, col_key in zip(axes, ['CPC', 'IMERG v07'], ['cpc', 'imerg']):
    for k, (name, spec) in enumerate(DISTRIBUTIONS.items()):
        delta_aics = []
        for entry in results:
            r = entry[col_key]
            if r['fits']:
                best_aic = min(f['aic'] for f in r['fits'].values())
                d = r['fits'][name]['aic'] - best_aic if name in r['fits'] else np.nan
            else:
                d = np.nan
            delta_aics.append(d)
        ax.bar(x_ticks + (k - 1.5) * width, delta_aics, width,
               label=name, color=spec['color'], alpha=0.85, edgecolor='white')

    ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([e['loc']['label'] for e in results], rotation=25, ha='right', fontsize=9)
    ax.set_ylabel('ΔAIC relative to best (lower = better)', fontsize=9)
    ax.set_title(ds_label, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_ylim(bottom=0)

fig.suptitle(
    'AIC relative to best-fit distribution per location\n'
    '(bar height 0 = winner;  taller bar = worse fit)',
    fontsize=12, fontweight='bold',
)
plt.tight_layout()
plt.show()

---
## 6. Single-location deep dive: CPC vs IMERG

Pick any location and compare how the two datasets look for the same place.

In [ ]:
# ── CHANGE THESE ──────────────────────────────────────────────────────────
LAT   = 39.0
LON   = -94.5
LABEL = 'Kansas City, MO'
# ──────────────────────────────────────────────────────────────────────────

rc = fit_point(LAT, LON, da_cpc,   ds_cpc)
ri = fit_point(LAT, LON, da_imerg, ds_imerg)

print(f"Location: {LABEL}  ({rc['lat']:.2f}°N, {rc['lon']:.2f}°E)")
print()
print('── CPC (gauge-based) ────────────────────────────────────────')
print_fit_table(rc, 'CPC')
print('── IMERG v07 (satellite) ────────────────────────────────────')
print_fit_table(ri, 'IMERG')

In [ ]:
# ── 4-panel: CPC (left column) vs IMERG (right column) ──────────────────
fig = plt.figure(figsize=(16, 11))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.40, wspace=0.30)

# Panel labels and (row, col) indices
panels = [
    (0, 0, 'A.  CPC — PDF (linear)',  rc, 'linear'),
    (0, 1, 'B.  IMERG — PDF (linear)', ri, 'linear'),
    (1, 0, 'C.  CPC — PDF (log scale)', rc, 'log'),
    (1, 1, 'D.  IMERG — PDF (log scale)', ri, 'log'),
]

for row, col, title, r, scale in panels:
    ax  = fig.add_subplot(gs[row, col])
    wet = r['wet']
    x   = _x_range(wet)
    fill_color = '#c8dff0' if r is rc else '#ffe5b4'

    ax.hist(wet, bins=40, density=True, color=fill_color,
            edgecolor='white', linewidth=0.4, label='Empirical', zorder=1)

    for name, spec in DISTRIBUTIONS.items():
        if name not in r['fits']:
            continue
        f   = r['fits'][name]
        pdf = spec['dist'].pdf(x, *f['params'])
        lw  = 2.8 if name == r['best'] else 1.6
        lbl = f"{name}" + (' ★' if name == r['best'] else '')
        ax.plot(x, pdf, color=spec['color'], ls=spec['ls'],
                lw=lw, label=lbl, zorder=2)

    if scale == 'log':
        ax.set_yscale('log')
    ax.set_xlabel('7-day mean precip (mm/day)', fontsize=9)
    ax.set_ylabel('Density' + (' (log)' if scale == 'log' else ''), fontsize=9)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.legend(fontsize=8, framealpha=0.9)
    ax.set_xlim(left=0)

fig.suptitle(
    f"{LABEL}  ({rc['lat']:.2f}°N, {rc['lon']:.2f}°E)\n"
    f"CPC: n={rc['n']}  best={rc['best']}  |  "
    f"IMERG: n={ri['n']}  best={ri['best']}",
    fontsize=12, fontweight='bold', y=1.01,
)
plt.show()

In [ ]:
# ── CDF + QQ side-by-side ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, r, ds_label, fill_c in zip(
    axes, [rc, ri], ['CPC', 'IMERG'], ['#2166ac', '#d6604d']
):
    wet  = r['wet']
    xs   = np.sort(wet)
    ecdf = np.arange(1, len(xs) + 1) / len(xs)
    x    = _x_range(wet)

    # CDF
    ax.step(xs, ecdf, color='black', lw=1.8, label='Empirical', zorder=3)
    for name, spec in DISTRIBUTIONS.items():
        if name not in r['fits']:
            continue
        f   = r['fits'][name]
        cdf = spec['dist'].cdf(x, *f['params'])
        lw  = 2.5 if name == r['best'] else 1.5
        ax.plot(x, cdf, color=spec['color'], ls=spec['ls'], lw=lw,
                label=f"{name} (KS={f['ks_stat']:.4f})" + (' ★' if name == r['best'] else ''))

    ax.set_xlabel('7-day mean precip (mm/day)', fontsize=9)
    ax.set_ylabel('Cumulative probability', fontsize=9)
    ax.set_title(f'CDF — {ds_label}', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8, framealpha=0.9)
    ax.set_xlim(left=0)
    ax.set_ylim(0, 1)

fig.suptitle(f'CDF comparison — {LABEL}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Q-Q plot: CPC vs IMERG using their respective best-fit distributions ──
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, r, ds_label in zip(axes, [rc, ri], ['CPC', 'IMERG']):
    wet  = r['wet']
    xs   = np.sort(wet)
    ecdf = np.arange(1, len(xs) + 1) / len(xs)

    for name, spec in DISTRIBUTIONS.items():
        if name not in r['fits']:
            continue
        f   = r['fits'][name]
        tq  = spec['dist'].ppf(ecdf, *f['params'])
        lw  = 2.5 if name == r['best'] else 1.0
        ms  = 3   if name == r['best'] else 2
        ax.scatter(tq, xs, s=ms, alpha=0.5, color=spec['color'],
                   label=name + (' ★' if name == r['best'] else ''), zorder=2)

    lim = xs.max() * 1.1
    ax.plot([0, lim], [0, lim], 'k--', lw=1.2, label='Perfect fit')
    ax.set_xlabel('Theoretical quantile (mm/day)', fontsize=9)
    ax.set_ylabel('Empirical quantile (mm/day)', fontsize=9)
    ax.set_title(f'Q-Q plot — {ds_label}', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8, framealpha=0.9)

fig.suptitle(f'Q-Q plots — {LABEL}\n(dots close to diagonal = good fit)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Return period curves: CPC vs IMERG

For the same location, compare how the two datasets extrapolate to rare events.  
Differences here reveal the **satellite vs gauge climatology gap**.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

dataset_style = [
    (rc, 'CPC',   '--', '#2166ac'),
    (ri, 'IMERG', '-',  '#d6604d'),
]

for r, ds_label, emp_ls, emp_color in dataset_style:
    wet = r['wet']
    xs  = np.sort(wet)[::-1]
    exc = np.arange(1, len(xs) + 1) / (len(xs) + 1)
    rp  = 1.0 / exc

    # Empirical
    ax.scatter(rp, xs, s=4, alpha=0.4, color=emp_color, zorder=2)

    # Best-fit theoretical curve (just the best-fit distribution)
    best_name = r['best']
    if best_name and best_name in r['fits']:
        spec = DISTRIBUTIONS[best_name]
        f    = r['fits'][best_name]
        x_plot = np.linspace(wet.min(), wet.max() * 1.6, 600)
        exc_fit = 1 - spec['dist'].cdf(x_plot, *f['params'])
        valid   = exc_fit > 1e-6
        rp_fit  = 1.0 / exc_fit[valid]
        ax.plot(rp_fit, x_plot[valid], color=emp_color, ls=emp_ls, lw=2.2,
                label=f"{ds_label} — {best_name} (n={r['n']})", zorder=3)

ax.set_xscale('log')
ax.set_xlabel('Return period (7-day windows)', fontsize=10)
ax.set_ylabel('7-day mean precipitation (mm/day)', fontsize=10)
ax.set_title(f'Return period curves — {LABEL}\nBest-fit distribution per dataset',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3)

ax2 = ax.twiny()
ax2.set_xscale('log')
ax2.set_xlim([v / 52 for v in ax.get_xlim()])
ax2.set_xlabel('Return period (years)', fontsize=9)

plt.tight_layout()
plt.show()

---
## 8. Pearson III skewness map — physical interpretation

The **Pearson III skewness parameter** captures how irregular rainfall is:
- **High skewness** → strongly right-skewed → dominated by rare intense events (convective belt, Gulf Coast)
- **Low skewness** → more symmetric / Gaussian-like → steady stratiform rainfall (Pacific NW)

This map reveals the **physical precipitation regimes** beneath the statistical pattern.

In [ ]:
# P3 skewness map: CPC vs IMERG
# Skewness > 0 = right-skewed (rare intense events dominate); 0 = near-symmetric
# Clipped to [0, 4]: negative values (fitting artefacts) mapped to minimum colour.
fig, axes = plt.subplots(
    1, 2, figsize=(20, 6),
    subplot_kw={'projection': ccrs.PlateCarree()},
)

cmap_skew = plt.get_cmap('YlOrRd')
cmap_skew.set_bad('lightgrey')

for ax, ds_s, title in zip(axes, [ds_cpc, ds_imerg], ['CPC (gauge-based)', 'IMERG v07 (satellite)']):
    ax = _conus_ax(ax)
    if 'pearson3_skew' not in ds_s:
        ax.set_title(f'{title}\n(pearson3_skew not in stats — re-run pipeline)', fontsize=10)
        continue
    data = ds_s['pearson3_skew'].values.copy()
    mask = (ds_s['n_wetdays'].values < MIN_SAMPLES) | ~np.isfinite(data)
    data[mask] = np.nan
    data = np.clip(data, 0, 4)
    im = ax.pcolormesh(ds_s.lon.values, ds_s.lat.values, data,
                       transform=ccrs.PlateCarree(), cmap=cmap_skew, vmin=0, vmax=4)
    cb = fig.colorbar(im, ax=ax, orientation='horizontal', pad=0.03, shrink=0.85)
    cb.set_label('Pearson III skewness  (0 = near-symmetric → 4 = strongly convective/extreme)', fontsize=9)
    cb.ax.tick_params(labelsize=8)
    ax.set_title(title, fontsize=12, fontweight='bold')

fig.suptitle(
    'Pearson III skewness parameter — spatial patterns reveal precipitation regimes\n'
    'Yellow = low skewness (stratiform/orographic)  ·  Red = high skewness (convective/extreme-dominated)',
    fontsize=12, fontweight='bold', y=1.02,
)
plt.tight_layout()
plt.show()

In [ ]:
# ── SHASH τ (tailweight) map: CPC vs IMERG ────────────────────────────────
# J&P formula, geosciences convention: τ = 1/δ_J&P
#   τ < 1 → lighter tails than normal;  τ = 1 → normal;  τ > 1 → heavier tails
# Range 0.5–3.5: yellow ≈ near-normal tails (τ~0.5–1), red = very heavy tails (τ~3.5+)
# A grid cell needs genuinely heavy AND flexible tails for SHASH to beat P3.
fig, axes = plt.subplots(
    1, 2, figsize=(20, 6),
    subplot_kw={'projection': ccrs.PlateCarree()},
)

cmap_tau = plt.get_cmap('YlOrRd')
cmap_tau.set_bad('lightgrey')

for ax, ds_s, title in zip(axes, [ds_cpc, ds_imerg], ['CPC (gauge-based)', 'IMERG v07 (satellite)']):
    ax = _conus_ax(ax)
    if 'shash_tailweight' not in ds_s:
        ax.set_title(f'{title}\n(shash_tailweight not in stats — re-run pipeline)', fontsize=10)
        continue
    data = ds_s['shash_tailweight'].values.copy()
    mask = (ds_s['n_wetdays'].values < MIN_SAMPLES) | ~np.isfinite(data)
    data[mask] = np.nan
    data = np.clip(data, 0.5, 3.5)
    im = ax.pcolormesh(ds_s.lon.values, ds_s.lat.values, data,
                       transform=ccrs.PlateCarree(), cmap=cmap_tau, vmin=0.5, vmax=3.5)
    cb = fig.colorbar(im, ax=ax, orientation='horizontal', pad=0.03, shrink=0.85)
    # Mark τ=1 (normal baseline) on the colorbar
    cb.ax.axvline((1.0 - 0.5) / (3.5 - 0.5), color='black', linewidth=1.5, linestyle='--')
    cb.ax.text((1.0 - 0.5) / (3.5 - 0.5), 1.08, 'τ=1\n(normal)',
               transform=cb.ax.transAxes, ha='center', va='bottom', fontsize=7, color='black')
    cb.set_label('SHASH τ (tailweight):  yellow=near-normal tails  →  red=very heavy tails', fontsize=9)
    cb.ax.tick_params(labelsize=8)
    ax.set_title(title, fontsize=12, fontweight='bold')

fig.suptitle(
    'SHASH tailweight parameter (τ)  —  Jones & Pewsey (2009) formula\n'
    'Cells with τ >> 1 have heavier tails than log-normal or gamma can capture  '
    '(dashed line = τ=1, normal baseline)',
    fontsize=12, fontweight='bold', y=1.02,
)
plt.tight_layout()
plt.show()

---
## 9. Distribution hierarchy and SHASH parameter space

### Why these four distributions?

The four families form a hierarchy from least to most flexible.

```
SHASH  (ε, τ, μ, σ — 4 params)          most general
 ├─ τ=1, ε=0  →  exactly Normal
 ├─ τ=1       →  log-normal family (log-scale normal; ε shifts log-space mean)
 └─ general τ, ε  →  can approximate Pearson III for the right (ε, τ) combination

Pearson III  (skew, loc, scale — 3 params)
 └─ loc=0  →  Gamma  (shape, scale — 2 params)
              └─ shape α=1  →  Exponential  (1 param)
```

**Key insight — why the four-way comparison matters:**  
P3 and log-normal each trace a **1-D curve** in skewness–kurtosis space (the Pearson Type III
locus and the log-normal locus, respectively).  SHASH lives in a **2-D space** (ε controls
skewness, τ controls tail-weight independently).  A grid cell where the data sits *off* both
1-D loci can only be captured by SHASH.

**Physical interpretation:**
| Distribution | Wins when … |
|---|---|
| Log-normal / Gamma | Moderate tails, fixed skew–kurtosis trade-off |
| Pearson III | Right-skewed, heavy tails (matches P3 locus well) |
| **SHASH** | **Extreme tails that exceed what P3 can represent at the observed skewness** — typically intense convection captured by IMERG |

The scatter plot below shows where the fitted (ε, τ) pairs fall for each dataset, coloured by
which distribution achieved the lowest AIC.  SHASH-winning cells should cluster at large τ,
confirming they need independently heavier tails than P3 permits.

In [ ]:
# ── SHASH parameter space: ε (skewness) vs τ (tailweight), coloured by best fit ──
# Each dot = one grid cell.  Dashed lines mark the "normal" baseline (τ=1, ε=0).
# SHASH-winning cells (orange) should cluster at large τ — they need heavier tails
# than P3 can provide at the observed skewness.

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, ds_s, title in zip(axes, [ds_cpc, ds_imerg], ['CPC (gauge-based)', 'IMERG v07 (satellite)']):
    mask = (ds_s['n_wetdays'].values >= MIN_SAMPLES) & np.isfinite(ds_s['shash_tailweight'].values)
    eps_all  = ds_s['shash_eps'].values[mask]
    tau_all  = ds_s['shash_tailweight'].values[mask]
    best_all = ds_s['best_fit_4way'].values[mask]

    # Plot each distribution's winning cells (smallest first so SHASH sits on top)
    for i, (name, color) in enumerate(zip(DIST_NAMES, DIST_COLORS)):
        idx = best_all == i
        if idx.sum() == 0:
            continue
        # Sub-sample large datasets to keep plot responsive
        step = max(1, idx.sum() // 8000)
        ax.scatter(eps_all[idx][::step], tau_all[idx][::step],
                   c=color, s=2 if i < 2 else 3, alpha=0.3 if i < 2 else 0.6,
                   label=f'{name} ({idx.sum():,})', rasterized=True, zorder=i + 1)

    # Reference lines: τ=1 (normal baseline) and ε=0 (symmetric)
    ax.axhline(1.0, color='black',  lw=1.2, ls='--', zorder=10, label='τ=1 (normal tails)')
    ax.axvline(0.0, color='dimgrey', lw=0.8, ls=':',  zorder=10, label='ε=0 (symmetric)')

    ax.set_xlabel('ε  (skewness parameter: ε>0 = right-skewed)', fontsize=10)
    ax.set_ylabel('τ  (tailweight: τ=1 = normal, τ>1 = heavier)', fontsize=10)
    ax.set_xlim(-3, 6)
    ax.set_ylim(0.3, 5)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, markerscale=4, loc='upper right', framealpha=0.9)

fig.suptitle(
    'SHASH parameter space (ε, τ)  —  coloured by best-fit distribution\n'
    'SHASH wins (orange) at large τ: data requires heavier tails than P3\'s skew–kurtosis constraint allows',
    fontsize=12, fontweight='bold',
)
plt.tight_layout()
plt.show()

# ── Median (ε, τ) by winning distribution ───────────────────────────────────
print('Median SHASH parameters by best-fit distribution:')
print(f"{'Distribution':<14}  {'Dataset':<8}  {'median ε':>9}  {'median τ':>9}  {'n cells':>8}")
print('-' * 60)
for ds_s, label in [(ds_cpc, 'CPC'), (ds_imerg, 'IMERG')]:
    mask = (ds_s['n_wetdays'].values >= MIN_SAMPLES) & np.isfinite(ds_s['shash_tailweight'].values)
    eps_a = ds_s['shash_eps'].values[mask]
    tau_a = ds_s['shash_tailweight'].values[mask]
    best  = ds_s['best_fit_4way'].values[mask]
    for i, name in enumerate(DIST_NAMES):
        idx = best == i
        if idx.sum() == 0:
            continue
        me = np.nanmedian(eps_a[idx])
        mt = np.nanmedian(tau_a[idx])
        print(f'{name:<14}  {label:<8}  {me:>9.2f}  {mt:>9.2f}  {idx.sum():>8,}')

---
## 10. Map: where do marked locations sit in the distribution landscape?

Overlay the 6 regional locations on the 4-way best-fit map.

In [ ]:
fig, axes = plt.subplots(
    1, 2, figsize=(20, 6),
    subplot_kw={'projection': ccrs.PlateCarree()},
)

for ax, ds_s, title in zip(axes, [ds_cpc, ds_imerg], ['CPC', 'IMERG v07']):
    ax = _conus_ax(ax)
    data = ds_s['best_fit_4way'].values.astype(float)
    data[ds_s['n_wetdays'].values < MIN_SAMPLES] = np.nan
    cmap4 = mcolors.ListedColormap(DIST_COLORS)
    norm4 = mcolors.BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap4.N)
    cmap4.set_bad('lightgrey')
    ax.pcolormesh(ds_s.lon.values, ds_s.lat.values, data,
                  transform=ccrs.PlateCarree(), cmap=cmap4, norm=norm4, zorder=1)

    # Marker colour = best-fit distribution colour for that dataset
    for entry in results:
        loc  = entry['loc']
        r    = entry['cpc'] if ds_s is ds_cpc else entry['imerg']
        best = r.get('best', 'Pearson III')
        lc   = DIST_COLORS[DIST_IDX.get(best, 2)]   # fallback: P3 green
        ax.scatter(loc['lon'], loc['lat'], transform=ccrs.PlateCarree(),
                   s=140, color=lc, edgecolors='white', linewidths=1.8, zorder=6)
        ax.text(loc['lon'] + 0.6, loc['lat'] + 0.4,
                f"{loc['label']}\n({best})",
                transform=ccrs.PlateCarree(), fontsize=7, color='white',
                fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor=lc, alpha=0.90),
                zorder=7)

    ax.set_title(title, fontsize=12, fontweight='bold')

# Shared distribution colorbar
sm = plt.cm.ScalarMappable(cmap=mcolors.ListedColormap(DIST_COLORS),
                            norm=mcolors.BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], 4))
cb = fig.colorbar(sm, ax=axes, orientation='horizontal', pad=0.04, shrink=0.4,
                  ticks=[0, 1, 2, 3])
cb.ax.set_xticklabels(DIST_NAMES, fontsize=9)

fig.suptitle(
    'Best-fit distribution map with selected analysis locations\n'
    '(marker colour = best-fit distribution at that point)',
    fontsize=12, fontweight='bold', y=1.02,
)
plt.show()